In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

### 1. Load data + define targets

In [ ]:
df = pd.read_csv("cleaned_data.csv")
print(f"Loaded cleaned_data.csv -> shape: {df.shape}")

TARGET_REG = "SalePrice"

y_reg = df[TARGET_REG].astype(float)
y_clf = (y_reg > y_reg.median()).astype(int)   # 1 = above-median price, 0 = below/equal

X = df.drop(columns=[TARGET_REG])

print(f"\ny_reg (SalePrice) summary:\n{y_reg.describe()}")
print(f"\ny_clf class balance (whole dataset):\n{y_clf.value_counts(normalize=True)}")

### 2. Encode categorical columns

In [ ]:
QUALITY_MAP = {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4}
ORDINAL_QUALITY_COLS = [
    "Exter_Qual", "Exter_Cond", "Bsmt_Qual", "Bsmt_Cond",
    "Heating_QC", "Kitchen_Qual", "Garage_Qual", "Garage_Cond",
]

for col in ORDINAL_QUALITY_COLS:
    X[col] = X[col].map(QUALITY_MAP)

ORDINAL_OTHER_MAPS = {
    "Lot_Shape":     {"Reg": 3, "IR1": 2, "IR2": 1, "IR3": 0},               # Regular -> most irregular
    "Land_Slope":    {"Gtl": 2, "Mod": 1, "Sev": 0},                          # Gentle -> Severe
    "Bsmt_Exposure": {"Gd": 3, "Av": 2, "Mn": 1, "No": 0},
    "BsmtFin_Type_1":{"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0},
    "BsmtFin_Type_2":{"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0},
    "Garage_Finish": {"Fin": 2, "RFn": 1, "Unf": 0},
    "Paved_Drive":   {"Y": 2, "P": 1, "N": 0},
    "Utilities":     {"AllPub": 2, "NoSewr": 1, "NoSeWa": 0},
    "Functional":    {"Typ": 7, "Min1": 6, "Min2": 5, "Mod": 4, "Maj1": 3, "Maj2": 2, "Sev": 1, "Sal": 0},
    "Central_Air":   {"Y": 1, "N": 0},
}
for col, mapping in ORDINAL_OTHER_MAPS.items():
    X[col] = X[col].map(mapping)

ALL_ORDINAL_COLS = ORDINAL_QUALITY_COLS + list(ORDINAL_OTHER_MAPS.keys())

nominal_cols = [c for c in X.columns if X[c].dtype == object or str(X[c].dtype).lower() == "str"]
print(f"\nOrdinal (label-encoded) columns: {len(ALL_ORDINAL_COLS)}")
print(f"Nominal (one-hot) columns: {len(nominal_cols)} -> {nominal_cols}")

X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

X = X.fillna(X.median(numeric_only=True))
X = X.astype(float)

print(f"\nFinal feature matrix shape after encoding: {X.shape}")

### 3. Leak-free train/test split + scaling

In [ ]:
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
scaler.fit(X_train)                      # fit ONLY on training data -> avoids leakage
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}")
print("NOTE: scaler was fit on X_train ONLY. Fitting on the full dataset would leak "
      "test-set mean/variance into the training pipeline (data leakage).")

#### 4. Regression — Linear Regression

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_reg_train)
y_pred_reg = lin_reg.predict(X_test_scaled)

mse_lin = mean_squared_error(y_reg_test, y_pred_reg)
r2_lin = r2_score(y_reg_test, y_pred_reg)
print(f"\n=== Linear Regression ===\nMSE: {mse_lin:,.2f}\nR2: {r2_lin:.4f}")

coef_df = pd.DataFrame({"feature": X.columns, "coefficient": lin_reg.coef_})
coef_df["abs_coef"] = coef_df["coefficient"].abs()
top3_lin = coef_df.sort_values("abs_coef", ascending=False).head(3)
print("\nTop 3 |coefficient| features (Linear Regression):")
print(top3_lin[["feature", "coefficient"]].to_string(index=False))

### Ridge Regression

In [ ]:
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge_reg.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)
print(f"\n=== Ridge Regression (alpha=1.0) ===\nMSE: {mse_ridge:,.2f}\nR2: {r2_ridge:.4f}")

print("\n--- OLS vs Ridge comparison table (copy into README) ---")
comp_table = pd.DataFrame({
    "Model": ["Linear Regression (OLS)", "Ridge (alpha=1.0)"],
    "MSE": [mse_lin, mse_ridge],
    "R2": [r2_lin, r2_ridge],
})
print(comp_table.to_string(index=False))

### 5. Classification — Logistic Regression

In [ ]:
class_counts_before = y_clf_train.value_counts().sort_index()
minority_pct = class_counts_before.min() / class_counts_before.sum() * 100
print(f"\n=== Class balance check: BEFORE any imbalance handling (y_clf_train) ===")
print(class_counts_before)
print(f"Minority class %: {minority_pct:.2f}%")

if minority_pct < 35:
    print("Imbalance detected (<35%) -> applying class_weight='balanced'.")
    use_balanced = True
else:
    print("Classes are already close to balanced (median split gives ~50/50) "
          "-> no resampling (e.g. SMOTE) is needed. class_weight='balanced' is "
          "still applied as a light-touch safeguard in case the split shifts "
          "the ratio slightly.")
    use_balanced = True

class_weight_arg = "balanced" if use_balanced else None
class_counts_after = y_clf_train.value_counts().sort_index() 
print(f"\n=== Class balance check: AFTER imbalance handling decision ===")
print(class_counts_after)
if class_counts_before.equals(class_counts_after):
    print("Counts are identical to 'before' -> confirms class_weight='balanced' "
          "re-weights the loss function during training rather than altering the "
          "training rows themselves (unlike SMOTE, which would change these counts).")

log_reg_baseline = LogisticRegression(max_iter=1000, C=1.0, class_weight=class_weight_arg,
                                       random_state=RANDOM_STATE)
log_reg_baseline.fit(X_train_scaled, y_clf_train)

y_pred_clf = log_reg_baseline.predict(X_test_scaled)
y_proba_baseline = log_reg_baseline.predict_proba(X_test_scaled)[:, 1]

cm = confusion_matrix(y_clf_test, y_pred_clf)
report = classification_report(y_clf_test, y_pred_clf)
print(f"\n=== Logistic Regression (C=1.0) ===\nConfusion matrix:\n{cm}")
print(f"\nClassification report:\n{report}")

fpr, tpr, thresholds_roc = roc_curve(y_clf_test, y_proba_baseline)
auc_baseline = roc_auc_score(y_clf_test, y_proba_baseline)
print(f"AUC (C=1.0): {auc_baseline:.4f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc_baseline:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Logistic Regression (C=1.0)")
plt.annotate(f"AUC = {auc_baseline:.3f}", xy=(0.6, 0.2))
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_curve_baseline.png"), dpi=150)
plt.close()
print(f"Saved ROC curve -> {OUT_DIR}/roc_curve_baseline.png")

### 5b. Decision-threshold sensitivity (0.30 - 0.70)


In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_rows = []
for t in thresholds:
    preds_t = (y_proba_baseline >= t).astype(int)
    p = precision_score(y_clf_test, preds_t, zero_division=0)
    r = recall_score(y_clf_test, preds_t, zero_division=0)
    f1 = f1_score(y_clf_test, preds_t, zero_division=0)
    threshold_rows.append({"Threshold": t, "Precision": p, "Recall": r, "F1": f1})

threshold_df = pd.DataFrame(threshold_rows)
print("\n=== Threshold sensitivity table (0.30 - 0.70) ===")
print(threshold_df.to_string(index=False))
best_threshold_row = threshold_df.loc[threshold_df["F1"].idxmax()]
print(f"\nF1-maximising threshold: {best_threshold_row['Threshold']} "
      f"(F1={best_threshold_row['F1']:.4f})")

### 6. Regularization experiment: C=0.01 vs C=1.0

In [ ]:
log_reg_strong_reg = LogisticRegression(max_iter=1000, C=0.01, class_weight=class_weight_arg,
                                         random_state=RANDOM_STATE)
log_reg_strong_reg.fit(X_train_scaled, y_clf_train)

y_pred_c001 = log_reg_strong_reg.predict(X_test_scaled)
y_proba_c001 = log_reg_strong_reg.predict_proba(X_test_scaled)[:, 1]

precision_baseline = precision_score(y_clf_test, y_pred_clf)
recall_baseline = recall_score(y_clf_test, y_pred_clf)

precision_c001 = precision_score(y_clf_test, y_pred_c001)
recall_c001 = recall_score(y_clf_test, y_pred_c001)
auc_c001 = roc_auc_score(y_clf_test, y_proba_c001)

print("\n=== Regularization comparison: C=1.0 vs C=0.01 ===")
reg_comp_table = pd.DataFrame({
    "Model": ["Logistic Regression (C=1.0)", "Logistic Regression (C=0.01)"],
    "Precision": [precision_baseline, precision_c001],
    "Recall": [recall_baseline, recall_c001],
    "AUC": [auc_baseline, auc_c001],
})
print(reg_comp_table.to_string(index=False))

### 7. Bootstrap confidence interval for AUC difference (C=1.0 minus C=0.01)

In [ ]:
N_BOOTSTRAP = 500
y_clf_test_arr = y_clf_test.to_numpy()
n_test = len(y_clf_test_arr)

boot_diffs = np.empty(N_BOOTSTRAP)
rng = np.random.RandomState(RANDOM_STATE)   # reproducible bootstrap

for i in range(N_BOOTSTRAP):
    idx = rng.choice(n_test, size=n_test, replace=True)   # use seeded rng, not global state
    y_sample = y_clf_test_arr[idx]

    # Skip degenerate resamples that contain only one class (AUC undefined)
    if len(np.unique(y_sample)) < 2:
        boot_diffs[i] = np.nan
        continue

    proba_baseline_sample = y_proba_baseline[idx]
    proba_c001_sample = y_proba_c001[idx]

    auc_baseline_sample = roc_auc_score(y_sample, proba_baseline_sample)
    auc_c001_sample = roc_auc_score(y_sample, proba_c001_sample)

    boot_diffs[i] = auc_baseline_sample - auc_c001_sample

valid_diffs = boot_diffs[~np.isnan(boot_diffs)]
mean_diff = np.mean(valid_diffs)
ci_lower = np.percentile(valid_diffs, 2.5)
ci_upper = np.percentile(valid_diffs, 97.5)

print(f"\n=== Bootstrap AUC difference (C=1.0 minus C=0.01), n={len(valid_diffs)} valid samples ===")
print(f"Mean AUC difference: {mean_diff:.4f}")
print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

excludes_zero = (ci_lower > 0) or (ci_upper < 0)
print(f"95% CI excludes zero: {excludes_zero}")
if excludes_zero:
    if mean_diff > 0:
        print("-> The interval is entirely above zero: C=1.0 reliably outperforms C=0.01 "
              "on AUC across resamples of the test set.")
    else:
        print("-> The interval is entirely below zero: C=0.01 (stronger regularization) "
              "reliably outperforms C=1.0 on AUC across resamples of the test set. "
              "This suggests the baseline model was mildly overfitting and the extra "
              "L2 penalty improved generalization on this dataset.")
else:
    print("-> The 95% CI includes zero, so the AUC difference between the two models "
          "may not be statistically reliable.")

In [ ]:
comp_table.to_csv(os.path.join(OUT_DIR, "ols_vs_ridge.csv"), index=False)
threshold_df.to_csv(os.path.join(OUT_DIR, "threshold_sensitivity.csv"), index=False)
reg_comp_table.to_csv(os.path.join(OUT_DIR, "regularization_comparison.csv"), index=False)
pd.DataFrame({
    "class": class_counts_before.index,
    "count_before": class_counts_before.values,
    "count_after": class_counts_after.values,
}).to_csv(os.path.join(OUT_DIR, "class_balance_before_after.csv"), index=False)
pd.DataFrame({
    "metric": ["mean_auc_diff", "ci_lower_2.5pct", "ci_upper_97.5pct", "excludes_zero"],
    "value": [mean_diff, ci_lower, ci_upper, excludes_zero],
}).to_csv(os.path.join(OUT_DIR, "bootstrap_auc_ci.csv"), index=False)

print(f"\nAll summary tables saved to ./{OUT_DIR}/ — use these to populate README.md")
print("Done.")